### Imports

In [ ]:
import rioxarray
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.colors as mcolors
import rasterio
from rasterio.features import rasterize
from shapely.geometry import box

In [ ]:
# 1. Define paths
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Inputs")
aboveground_global = base_path / "aboveground_biomass_carbon_2010.tif"
belowground_global = base_path / "belowground_biomass_carbon_2010.tif"
jamaica_boundary_path = base_path / "Boundaries/jamaica.gpkg"
land_use_path = base_path / "2013_landuse_LandCover.shp"

In [ ]:
# 2. Load Jamaica boundary
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
jam_crs = jamaica_boundary.crs
print("Jamaica boundary CRS:", jam_crs)

In [ ]:
# 3. Load and clip aboveground carbon raster
above_ras = rioxarray.open_rasterio(aboveground_global)
jamaica_boundary_raster_crs = jamaica_boundary.to_crs(above_ras.rio.crs)
above_clipped_native = above_ras.rio.clip(
    [jamaica_boundary_raster_crs.geometry.iloc[0]], above_ras.rio.crs, drop=True
)
above_clipped_reproj = above_clipped_native.rio.reproject(jam_crs)
above_clipped_out = base_path / "aboveground_carbon_jamaica_clipped_reproj.tif"
above_clipped_reproj.rio.to_raster(above_clipped_out)

In [ ]:
# 4. Load and clip belowground carbon raster
below_ras = rioxarray.open_rasterio(belowground_global)
jamaica_boundary_below_raster_crs = jamaica_boundary.to_crs(below_ras.rio.crs)
below_clipped_native = below_ras.rio.clip(
    [jamaica_boundary_below_raster_crs.geometry.iloc[0]], below_ras.rio.crs, drop=True
)
below_clipped_reproj = below_clipped_native.rio.reproject(jam_crs)
below_clipped_out = base_path / "belowground_carbon_jamaica_clipped_reproj.tif"
below_clipped_reproj.rio.to_raster(below_clipped_out)

In [ ]:
# 5. Mask NoData and plot
def process_and_plot_raster(raster_path, title, color_min, color_max, boundary, save_path):
    ras = rioxarray.open_rasterio(raster_path)
    masked_ras = ras.isel(band=0).where(ras.isel(band=0) != ras.rio.nodata)
    print(f"{title} - Min:", np.nanmin(masked_ras.values))
    print(f"{title} - Max:", np.nanmax(masked_ras.values))

    # Plot raster with boundary
    fig, ax = plt.subplots(figsize=(8, 8))
    masked_ras.plot.imshow(ax=ax, cmap="viridis", add_colorbar=True, vmin=color_min, vmax=color_max)
    boundary.boundary.plot(ax=ax, edgecolor="red", linewidth=2)
    ax.set_title(title)
    plt.show()

    # Save masked raster
    masked_ras.rio.to_raster(save_path)
    print(f"{title} - Masked raster saved as '{save_path}'")

process_and_plot_raster(
    raster_path=above_clipped_out,
    title="Aboveground Carbon - Jamaica",
    color_min=0,
    color_max=3000,
    boundary=jamaica_boundary,
    save_path="jamaica_aboveground_carbon_masked.tif"
)

process_and_plot_raster(
    raster_path=below_clipped_out,
    title="Belowground Carbon - Jamaica",
    color_min=0,
    color_max=750,
    boundary=jamaica_boundary,
    save_path="jamaica_belowground_carbon_masked.tif"
)


In [ ]:
from shapely.geometry import box

# 6. Zonal statistics by land use
# Load the land use layer and ensure CRS matches the raster CRS
terrestrial_landcover = gpd.read_file(land_use_path).to_crs(jam_crs)
print("Land use CRS:", terrestrial_landcover.crs)

# Filter land use polygons to ensure they intersect the raster bounds
raster_bounds = above_clipped_reproj.rio.bounds()  # (minx, miny, maxx, maxy)
raster_bbox = box(*raster_bounds)  # Create a bounding box geometry
valid_land_use = terrestrial_landcover[
    terrestrial_landcover.geometry.intersects(raster_bbox)
]

print(f"Filtered {len(valid_land_use)} valid polygons out of {len(terrestrial_landcover)}.")

In [ ]:
# Step 1: Add Area in Hectares to the Land Use DataFrame
valid_land_use["area_ha"] = valid_land_use.geometry.area / 10_000  # Convert m² to hectares
print(valid_land_use[["Classify", "area_ha"]].head())  # Replace "Classify" with your land use type column

# Step 2: Initialize a List for Zonal Statistics
zonal_stats_results = []

for _, row in valid_land_use.iterrows():
    land_use_geom = [row.geometry]
    try:
        # Clip the rasters to the polygon
        masked_above = above_clipped_reproj.rio.clip(land_use_geom, jam_crs, drop=True)
        masked_below = below_clipped_reproj.rio.clip(land_use_geom, jam_crs, drop=True)

        # Check if the rasters contain valid data
        if np.isnan(masked_above.values).all():
            above_sum = 0
        else:
            above_sum = np.nansum(masked_above.values)

        if np.isnan(masked_below.values).all():
            below_sum = 0
        else:
            below_sum = np.nansum(masked_below.values)

        # Add stats to the list
        stats = {
            "land_use_type": row["Classify"],  # Replace with your column name
            "area_ha": row["area_ha"],  # Area in hectares
            "above_sum": above_sum,
            "below_sum": below_sum,
            "above_per_ha": above_sum / row["area_ha"] if row["area_ha"] > 0 else 0,
            "below_per_ha": below_sum / row["area_ha"] if row["area_ha"] > 0 else 0,
        }
        zonal_stats_results.append(stats)

    except Exception as e:
        print(f"Error processing polygon {row['Classify']}: {e}")
        stats = {
            "land_use_type": row["Classify"],
            "area_ha": row["area_ha"],
            "above_sum": 0,
            "below_sum": 0,
            "above_per_ha": 0,
            "below_per_ha": 0,
        }
        zonal_stats_results.append(stats)

# Step 3: Convert Results to a DataFrame
zonal_stats_df = pd.DataFrame(zonal_stats_results)

# Step 4: Summarize by Land Use Type
land_use_summary = zonal_stats_df.groupby("land_use_type").agg({
    "area_ha": "sum",
    "above_sum": "sum",
    "below_sum": "sum",
    "above_per_ha": "mean",  # Average aboveground carbon per hectare
    "below_per_ha": "mean",  # Average belowground carbon per hectare
}).reset_index()

# Display the summary
print(land_use_summary)

# Save to CSV
land_use_summary.to_csv(base_path / "carbon_per_hectare_by_land_use.csv", index=False)
print("Summary saved as 'carbon_per_hectare_by_land_use.csv'.")

In [ ]:
# Group by land use type and calculate summary statistics
land_use_summary = zonal_stats_df.groupby("land_use_type").agg({
    "area_ha": "sum",        # Total area for each land use type
    "above_sum": "sum",      # Total aboveground carbon
    "below_sum": "sum",      # Total belowground carbon
}).reset_index()

# Calculate average carbon per hectare
land_use_summary["above_per_ha"] = land_use_summary["above_sum"] / land_use_summary["area_ha"]
land_use_summary["below_per_ha"] = land_use_summary["below_sum"] / land_use_summary["area_ha"]

# Display the summarized results
print(land_use_summary)

# Save to a CSV file
land_use_summary.to_csv(base_path / "grouped_carbon_by_land_use.csv", index=False)
print("Grouped carbon data saved as 'grouped_carbon_by_land_use.csv'.")

In [ ]:
land_use_summary["above_per_ha"] = land_use_summary["above_sum"] / land_use_summary["area_ha"]
land_use_summary["below_per_ha"] = land_use_summary["below_sum"] / land_use_summary["area_ha"]

fig, ax = plt.subplots(figsize=(10, 6))

# Stacked bar chart for total carbon (above and below ground)
land_use_summary.plot(
    x="land_use_type",
    y=["above_sum", "below_sum"],
    kind="bar",
    stacked=True,
    ax=ax,
    alpha=0.7,
)

# Customize the plot
ax.set_title("Total Carbon by Land Use Type")
ax.set_xlabel("Land Use Type")
ax.set_ylabel("Total Carbon (units)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Group by land use type and calculate summary statistics
land_use_summary = zonal_stats_df.groupby("land_use_type").agg({
    "area_ha": "sum",        # Total area for each land use type
    "above_sum": "sum",      # Total aboveground carbon
    "below_sum": "sum",      # Total belowground carbon
}).reset_index()

# Calculate average carbon per hectare
land_use_summary["above_per_ha"] = land_use_summary["above_sum"] / land_use_summary["area_ha"]
land_use_summary["below_per_ha"] = land_use_summary["below_sum"] / land_use_summary["area_ha"]

# Display the summarized results
print(land_use_summary)

# Save to a CSV file
land_use_summary.to_csv(base_path / "grouped_carbon_by_land_use.csv", index=False)
print("Grouped carbon data saved as 'grouped_carbon_by_land_use.csv'.")

In [ ]:
# Group by land_use_type and calculate the sum for each column
land_use_summary = zonal_stats_df.groupby("land_use_type").sum()

# Display the summarized results
print(land_use_summary)

In [ ]:
# Plot the aboveground and belowground carbon sums as a bar chart
fig, ax = plt.subplots(figsize=(10, 6))

# Plot aboveground carbon
ax.bar(land_use_summary.index, land_use_summary["above_sum"], label="Aboveground Carbon", alpha=0.7)

# Plot belowground carbon on the same axis
ax.bar(land_use_summary.index, land_use_summary["below_sum"], label="Belowground Carbon", alpha=0.7, bottom=land_use_summary["above_sum"])

# Customize the chart
ax.set_title("Carbon Storage by Land Use Type")
ax.set_xlabel("Land Use Type")
ax.set_ylabel("Total Carbon (units)")
ax.legend()
plt.xticks(rotation=45, ha="right")  # Rotate x-axis labels for better readability
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Step 1: Add Area in Hectares to the Land Use DataFrame
valid_land_use["area_ha"] = valid_land_use.geometry.area / 10_000  # Convert m² to hectares
print(valid_land_use[["Classify", "area_ha"]].head())  # Replace "Classify" with your land use type column

# Step 2: Initialize a List for Zonal Statistics
zonal_stats_results = []

for _, row in valid_land_use.iterrows():
    land_use_geom = [row.geometry]
    try:
        # Clip the rasters to the polygon
        masked_above = above_clipped_reproj.rio.clip(land_use_geom, jam_crs, drop=True)
        masked_below = below_clipped_reproj.rio.clip(land_use_geom, jam_crs, drop=True)

        # Check if the rasters contain valid data
        if np.isnan(masked_above.values).all():
            above_sum = 0
        else:
            above_sum = np.nansum(masked_above.values)

        if np.isnan(masked_below.values).all():
            below_sum = 0
        else:
            below_sum = np.nansum(masked_below.values)

        # Add stats to the list
        stats = {
            "land_use_type": row["Classify"],  # Replace with your column name
            "area_ha": row["area_ha"],  # Area in hectares
            "above_sum": above_sum,
            "below_sum": below_sum,
            "above_per_ha": above_sum / row["area_ha"] if row["area_ha"] > 0 else 0,
            "below_per_ha": below_sum / row["area_ha"] if row["area_ha"] > 0 else 0,
        }
        zonal_stats_results.append(stats)

    except Exception as e:
        print(f"Error processing polygon {row['Classify']}: {e}")
        stats = {
            "land_use_type": row["Classify"],
            "area_ha": row["area_ha"],
            "above_sum": 0,
            "below_sum": 0,
            "above_per_ha": 0,
            "below_per_ha": 0,
        }
        zonal_stats_results.append(stats)

# Step 3: Convert Results to a DataFrame
zonal_stats_df = pd.DataFrame(zonal_stats_results)

# Step 4: Summarize by Land Use Type
land_use_summary = zonal_stats_df.groupby("land_use_type").agg({
    "area_ha": "sum",
    "above_sum": "sum",
    "below_sum": "sum",
    "above_per_ha": "mean",  # Average aboveground carbon per hectare
    "below_per_ha": "mean",  # Average belowground carbon per hectare
}).reset_index()

# Display the summary
print(land_use_summary)

# Save to CSV
land_use_summary.to_csv(base_path / "carbon_per_hectare_by_land_use.csv", index=False)
print("Summary saved as 'carbon_per_hectare_by_land_use.csv'.")

In [ ]:
# Step 3: Define Raster Metadata
# Get bounds from land use layer and define resolution
minx, miny, maxx, maxy = terrestrial_landcover.total_bounds
resolution = 10  # Set your desired resolution (e.g., 10 meters per pixel)

# Define raster transform and dimensions
width = int((maxx - minx) / resolution)
height = int((maxy - miny) / resolution)
transform = rasterio.transform.from_origin(minx, maxy, resolution, resolution)

# Step 4: Rasterize the Land Use Polygons
# Use the "Classify" column as raster values (replace with your actual column name)
def get_terrestrial_landcover_value(row):
    # Map land use types to numeric values, e.g., {"Forest": 1, "Urban": 2}
    terrestrial_landcover_mapping = {"Forest": 1, "Urban": 2, "Agriculture": 3, "Other": 4}
    return terrestrial_landcover_mapping.get(row["Classify"], 0)  # Default to 0 for unknown types

# Apply the mapping to the GeoDataFrame
terrestrial_landcover["raster_value"] = terrestrial_landcover.apply(get_terrestrial_landcover_value, axis=1)

# Rasterize polygons
rasterized_terrestrial_landcover = rasterize(
    [(geom, value) for geom, value in zip(terrestrial_landcover.geometry, terrestrial_landcover["raster_value"])],
    out_shape=(height, width),
    transform=transform,
    fill=0,  # Background value for areas without data
    dtype="int32"
)


In [ ]:
# Step 5: Save the Rasterized Land Use File
output_raster_path = "/path/to/output_land_use_raster.tif"
with rasterio.open(
    output_raster_path,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,  # Single band
    dtype="int32",
    crs=target_crs,
    transform=transform,
) as dst:
    dst.write(rasterized_land_use, 1)  # Write to the first band

print(f"Land use raster saved to {output_raster_path}")

In [ ]:
# Step 1: Merge Land Use Data with Carbon Statistics
# Ensure "land_use_type" is the common column between valid_land_use and zonal_stats_df
valid_land_use = valid_land_use.merge(zonal_stats_df, on="land_use_type")

# Step 2: Create a Combined Carbon Metric
valid_land_use["total_carbon"] = valid_land_use["above_sum"] + valid_land_use["below_sum"]

# Step 3: Plot the Land Use Layer with Highlighted Carbon
fig, ax = plt.subplots(figsize=(12, 8))

# Create a color map for total carbon
norm = mcolors.Normalize(vmin=valid_land_use["total_carbon"].min(), vmax=valid_land_use["total_carbon"].max())
cmap = plt.cm.viridis

# Plot the land use polygons
valid_land_use.plot(
    ax=ax,
    column="total_carbon",  # Column to use for coloring
    cmap=cmap,              # Color map
    norm=norm,              # Normalization (color scaling)
    legend=True,            # Add a legend
    legend_kwds={"label": "Total Carbon (units)"}
)

# Customize the map
ax.set_title("Land Use and Highlighted Carbon Areas")
ax.set_axis_off()  # Remove axis for a cleaner map

# Show the map
plt.show()

In [ ]:
# Raster bounds
print("Aboveground raster bounds:", above_clipped_reproj.rio.bounds())
print("Belowground raster bounds:", below_clipped_reproj.rio.bounds())

# Land use bounds
print("Land use bounds:", terrestrial_landcover.total_bounds)

In [ ]:
# 7. Save the final clipped + reprojected Aboveground raster
above_clipped_out = base_path / "aboveground_carbon_jamaica_clipped_reproj.tif"
above_clipped_reproj.rio.to_raster(above_clipped_out)

In [ ]:
print("Aboveground subset size:", above_clipped_native.shape)
print("Aboveground reprojected subset size:", above_clipped_reproj.shape)

# Print CRS of the native-clipped subset
print("Aboveground reprojected subset CRS (final):", above_clipped_reproj.rio.crs)

In [ ]:
# Repeat the same process for Belowground
# ---------------------------
below_ras = rioxarray.open_rasterio(belowground_global)
print("Global Belowground Raster CRS:", below_ras.rio.crs)

In [ ]:
# Reproject the same boundary to match the below raster’s CRS
jamaica_boundary_below_raster_crs = jamaica_boundary.to_crs(below_ras.rio.crs)
jam_polygon_below_raster_crs = jamaica_gdf_below_raster_crs.geometry.iloc[0]

In [ ]:
# Clip in the original (global) CRS
below_clipped_native = below_ras.rio.clip([jam_polygon_below_raster_crs], below_ras.rio.crs, drop=True)

In [ ]:
# Now reproject the smaller area
below_clipped_reproj = below_clipped_native.rio.reproject(jam_crs)

In [ ]:
# Save final
below_clipped_out = base_path / "belowground_carbon_jamaica_clipped_reproj.tif"
below_clipped_reproj.rio.to_raster(below_clipped_out)

print("Belowground subset size:", below_clipped_native.shape)
print("Belowground reprojected subset size:", below_clipped_reproj.shape)

print("Belowground reprojected subset CRS (final):", below_clipped_reproj.rio.crs)

# Analyse data

In [ ]:
Jamaica_aboveground_carbon = base_path / "jamaica_aboveground_carbon_clipped_reproj.tif"
Jamaica_belowground_carbon = base_path / "jamaica_belowground_carbon_clipped_reproj.tif"

In [ ]:
# --- Load your clipped & reprojected raster ---

above_clipped_out = base_path / "aboveground_carbon_jamaica_clipped_reproj.tif"
print("Aboveground raster CRS:", above_ras.rio.crs)
print("NoData value (detected):", above_ras.rio.nodata)
print("Boundary CRS:", jamaica_boundary.crs)

# --- Step 1: Mask NoData values ---
masked_above_ras = above_ras.isel(band=0).where(above_ras.isel(band=0) != above_ras.rio.nodata)

# --- Step 2: Inspect data values ---
# Check range of values in the raster after masking
print("Masked Min Value:", np.nanmin(masked_above_ras.values))
print("Masked Max Value:", np.nanmax(masked_above_ras.values))

# --- Step 3: Define color scale ---
vmin, vmax = 0, 3000  # Adjust these values based on your dataset

# --- Step 4: Plot Raster and Boundary ---
fig, ax = plt.subplots(figsize=(8, 8))

# Plot the masked raster
masked_above_ras.plot.imshow(
    ax=ax,
    cmap="viridis",
    add_colorbar=True,
    vmin=vmin,
    vmax=vmax
)

# Overlay the Jamaica boundary
jamaica_boundary.boundary.plot(ax=ax, edgecolor="red", linewidth=2)

# Add title and show
ax.set_title("Aboveground Carbon - Jamaica (Masked + Boundary)")
ax.set_aspect("equal")  # Ensure square pixels
plt.show()

In [ ]:
# --- Load the Jamaica boundary ---
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print("Boundary CRS:", jamaica_boundary.crs)

# --- Helper Function to Process and Plot a Raster ---
def process_and_plot_raster(raster_path, title, color_min, color_max, boundary, save_path=None):
    # Load the raster
    ras = rioxarray.open_rasterio(raster_path)
    print(f"{title} - Raster CRS:", ras.rio.crs)
    print(f"{title} - NoData value (detected):", ras.rio.nodata)
    
    # Mask NoData values
    masked_ras = ras.isel(band=0).where(ras.isel(band=0) != ras.rio.nodata)
    
    # Check data range
    print(f"{title} - Min Value (masked):", np.nanmin(masked_ras.values))
    print(f"{title} - Max Value (masked):", np.nanmax(masked_ras.values))
    
    # Create a figure and axis for plotting
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Plot the masked raster
    masked_ras.plot.imshow(
        ax=ax,
        cmap="viridis",
        add_colorbar=True,
        vmin=color_min,
        vmax=color_max
    )
    
    # Overlay the boundary
    boundary.boundary.plot(ax=ax, edgecolor="red", linewidth=2)
    
    # Add title and show
    ax.set_title(title)
    ax.set_aspect("equal")  # Ensure square pixels
    plt.show()
    
    # Optionally save the masked raster
    if save_path:
        masked_ras.rio.to_raster(save_path)
        print(f"Masked raster saved as '{save_path}'")

# --- Plot Aboveground Carbon ---
process_and_plot_raster(
    raster_path=above_clipped_out,
    title="Aboveground Carbon - Jamaica",
    color_min=0,
    color_max=3000,  # Adjust range based on aboveground carbon data
    boundary=jamaica_boundary,
    save_path="jamaica_aboveground_carbon_masked.tif"
)

# --- Plot Belowground Carbon ---
process_and_plot_raster(
    raster_path=below_clipped_out,
    title="Belowground Carbon - Jamaica (Improved Colormap)",
    color_min=0,
    color_max=750,
    boundary=jamaica_boundary,
    save_path="jamaica_belowground_carbon_masked.tif"
)

In [ ]:
# --- Optional Step 5: Save Masked Raster ---
# Save the masked raster if needed
masked_above_ras.rio.to_raster("jamaica_aboveground_carbon_masked.tif")
print("Masked raster saved as 'jamaica_aboveground_carbon_masked.tif'")

In [ ]:
data_array = above_ras.isel(band=0)
vals = data_array.values  # This is a NumPy array (y, x)
mask = np.isnan(vals)

print("Min:", np.nanmin(vals[~mask]))
print("Max:", np.nanmax(vals[~mask]))
print("Mean:", np.nanmean(vals[~mask]))

In [ ]:
print("NoData value (detected by rioxarray):", above_ras.rio.nodata)

In [ ]:
# Analyze the belowground raster
below_ras = rioxarray.open_rasterio(below_clipped_out)
masked_below_ras = below_ras.isel(band=0).where(below_ras.isel(band=0) != below_ras.rio.nodata)

# Print the range of values
print("Belowground Carbon - Min Value (masked):", np.nanmin(masked_below_ras.values))
print("Belowground Carbon - Max Value (masked):", np.nanmax(masked_below_ras.values))

# Analyse carbon by land use

In [ ]:
land_use = base_path / "2013_landuse_LandCover.shp"

In [ ]:
terrestrial_landcover = gpd.read_file(land_use)
terrestrial_landcover = terrestrial_landcover.to_crs(jamaica_metric_grid_crs)
print(terrestrial_landcover.crs)

In [ ]:
# Initialize a dictionary to store results
zonal_stats_results = []

In [ ]:
# Loop through land use polygons
for _, row in land_use.iterrows():
    # Mask the raster to the current polygon
    masked_raster = raster.rio.clip([row.geometry], raster.rio.crs, drop=True)
    data = masked_raster.values[0]  # Extract values as a NumPy array

    # Compute statistics (ignoring NaNs)
    stats = {
        "land_use_type": row["land_use_type"],  # Replace with your land use column name
        "mean": np.nanmean(data),
        "sum": np.nansum(data),
        "min": np.nanmin(data),
        "max": np.nanmax(data)
    }
    zonal_stats_results.append(stats)

# Convert results to a pandas DataFrame
zonal_stats_df = pd.DataFrame(zonal_stats_results)
print(zonal_stats_df)